# Bài 38: Học chuyển giao (Transfer Learning)

Huấn luyện một CNN tốt từ đầu (Bài 34-35) cần hàng trăm mẫu có nhãn ngay cả cho một nhiệm vụ đồ chơi. Các nhiệm vụ đích thực tế thường đói dữ liệu: một nhúm ảnh chụp y tế có nhãn, một danh mục sản phẩm mới với 20 bức ảnh. **Học chuyển giao (transfer learning)** né tránh điều này bằng cách dùng lại một mạng đã được huấn luyện trên một nhiệm vụ *khác*, giàu dữ liệu, dựa trên giả thuyết rằng các đặc trưng ở lớp sớm (biên, đốm, kết cấu đơn giản) hữu ích cho hầu như mọi nhiệm vụ thị giác, không chỉ nhiệm vụ mà chúng được huấn luyện ban đầu. Bài này minh họa điều đó trên ảnh chụp thật, dùng một tập con CIFAR-10 (<a href="../references.html#krizhevsky-2009-cifar">Krizhevsky, 2009</a>).

In [ ]:
import copy
import pickle
import tarfile
import urllib.request
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Nhiệm vụ nguồn và nhiệm vụ đích (Source task and target task)
Thiết lập hai nhiệm vụ liên quan nhưng khác biệt, đều từ CIFAR-10. **Nhiệm vụ nguồn (source task)** có nhiều dữ liệu: phân biệt máy bay với ô tô, mỗi lớp 300 ảnh huấn luyện. **Nhiệm vụ đích (target task)** là nhiệm vụ ta thực sự quan tâm, và nó bị bỏ đói có chủ đích: phân biệt mèo với xe tải — hai lớp mà nhiệm vụ nguồn hoàn toàn chưa từng thấy — với chỉ **15 ảnh huấn luyện mỗi lớp**.

In [ ]:
CIFAR_URL = 'https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz'
CACHE_ROOT = Path.home() / '.cache' / 'cvintro'
CACHE_DIR = CACHE_ROOT / 'cifar-10-batches-py'

def ensure_cifar10():
    if CACHE_DIR.exists():
        return
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    archive_path = CACHE_ROOT / 'cifar-10-python.tar.gz'
    if not archive_path.exists():
        print('Downloading CIFAR-10 (~163 MB, one-time, cached under ~/.cache/cvintro)...')
        urllib.request.urlretrieve(CIFAR_URL, archive_path)
    print('Extracting...')
    with tarfile.open(archive_path) as tar:
        tar.extractall(CACHE_ROOT)

def load_cifar_batch(path):
    with open(path, 'rb') as f:
        d = pickle.load(f, encoding='bytes')
    imgs = d[b'data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1).astype(np.float32) / 255.0
    labels = np.array(d[b'labels'], dtype=np.int64)
    return imgs, labels

ensure_cifar10()

CIFAR_LABELS = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
CID = {name: CIFAR_LABELS.index(name) for name in CIFAR_LABELS}

train_imgs, train_labels = [], []
for i in range(1, 6):
    imgs, labels = load_cifar_batch(CACHE_DIR / f'data_batch_{i}')
    train_imgs.append(imgs); train_labels.append(labels)
train_imgs, train_labels = np.concatenate(train_imgs), np.concatenate(train_labels)
test_imgs, test_labels = load_cifar_batch(CACHE_DIR / 'test_batch')

def take(imgs, labels, name, n, rng_local):
    idx = np.where(labels == CID[name])[0]
    idx = rng_local.permutation(idx)[:n]
    return imgs[idx]

data_rng = np.random.default_rng(3)

# source task: airplane vs. automobile, plenty of data
src_a = take(train_imgs, train_labels, 'airplane', 300, data_rng)
src_b = take(train_imgs, train_labels, 'automobile', 300, data_rng)
X_src = np.concatenate([src_a, src_b])
y_src = np.concatenate([np.zeros(300), np.ones(300)]).astype(np.int64)
perm = data_rng.permutation(len(X_src)); X_src, y_src = X_src[perm], y_src[perm]

# target task: cat vs. truck, neither ever seen by the source task, deliberately starved
X_tgt_train = np.concatenate([take(train_imgs, train_labels, 'cat', 15, data_rng),
                               take(train_imgs, train_labels, 'truck', 15, data_rng)])
y_tgt_train = np.array([0] * 15 + [1] * 15, dtype=np.int64)
X_tgt_test = np.concatenate([take(test_imgs, test_labels, 'cat', 150, data_rng),
                              take(test_imgs, test_labels, 'truck', 150, data_rng)])
y_tgt_test = np.array([0] * 150 + [1] * 150, dtype=np.int64)

fig, axes = plt.subplots(2, 6, figsize=(11, 4))
for ax, im in zip(axes[0], list(src_a[:3]) + list(src_b[:3])):
    ax.imshow(im); ax.axis('off')
axes[0, 0].set_ylabel('source', rotation=0, labelpad=25)
for ax, im in zip(axes[1], list(X_tgt_train[:3]) + list(X_tgt_train[15:18])):
    ax.imshow(im); ax.axis('off')
axes[1, 0].set_ylabel('target', rotation=0, labelpad=25)
fig.suptitle('Source task (airplane vs. automobile, top) vs. target task (cat vs. truck, bottom)', y=1.02)
plt.show()

## Ba chiến lược (Three strategies)
1. **Từ đầu (from scratch)** — huấn luyện một CNN mới chỉ trên 30 ảnh đích. Đây là đường cơ sở: hoàn toàn không chuyển giao.
2. **Mạng xương sống đóng băng (frozen backbone — trích xuất đặc trưng)** — tiền huấn luyện (pretrain) một mạng xương sống CNN trên nhiệm vụ nguồn, rồi đóng băng hoàn toàn trọng số của nó và chỉ huấn luyện một bộ phân loại tuyến tính mới bên trên các đặc trưng mà nó tạo ra cho ảnh đích.
3. **Tinh chỉnh (fine-tuning)** — bắt đầu từ cùng mạng xương sống đã tiền huấn luyện, nhưng tiếp tục cập nhật nó trên dữ liệu đích, dùng tốc độ học *nhỏ hơn nhiều* cho mạng xương sống so với phần đầu phân loại (classifier head) mới (mạng xương sống đã mã hóa cấu trúc hữu ích; các cập nhật lớn chỉ từ 30 mẫu sẽ phá hỏng nó).

Ảnh động bên dưới cho thấy ba chiến lược cùng lúc: một lượt xuôi chảy từ trái sang phải qua mọi lớp giống hệt nhau (ô có viền đánh dấu lớp hiện tại), rồi trạng thái của mọi lớp được hiển thị cùng lúc &mdash; bông tuyết cho đóng băng, ngọn lửa cho huấn luyện được, kích thước theo tốc độ học &mdash; trước khi một lượt ngược chảy từ phải sang trái bắt đầu ở phần đầu, gần hàm mất mát nhất, nên nó cập nhật trước. Một ô tô đỏ có viền màu đánh dấu một lớp có trọng số thực sự được cập nhật khi gradient tới nó; các lớp bị đóng băng hoàn toàn không có đánh dấu như vậy, vì không có gì xảy ra với chúng.

<img src="../animations/finetune_lr_demo.gif" alt="Ảnh động lượt xuôi và lượt ngược qua ba chiến lược học chuyển giao xếp chồng từ trên xuống dưới: một vệt sáng lượt xuôi viền quanh mỗi ô từ trái sang phải qua input, conv1, conv2, conv3 và head giống hệt nhau ở cả ba hàng, rồi biểu tượng bông tuyết (đóng băng) hoặc ngọn lửa (huấn luyện được, kích thước theo tốc độ học) của mọi lớp xuất hiện cùng lúc và giữ trong giây lát, rồi một lượt ngược bắt đầu ở head và quét từ phải sang trái, chuyển ô của mỗi lớp được cập nhật thành màu đỏ có viền màu khi gradient tới nó -- từ đầu chuyển mọi ô thành đỏ, mạng xương sống đóng băng chỉ chuyển head thành đỏ để nguyên mọi lớp xương sống, tinh chỉnh chuyển mọi ô thành đỏ với ngọn lửa nhỏ trên mạng xương sống và ngọn lửa lớn trên head" style="display: block; margin: 0 auto; max-width: 820px;">

In [ ]:
class Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 16, 5, padding=2), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 5, padding=2), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.AdaptiveMaxPool2d(1),
        )

    def forward(self, x):
        return self.conv(x).flatten(1)

class Classifier(nn.Module):
    def __init__(self, backbone, freeze_backbone):
        super().__init__()
        self.backbone = backbone
        self.freeze_backbone = freeze_backbone
        self.fc = nn.Linear(32, 2)

    def forward(self, x):
        feat = self.backbone(x)
        if self.freeze_backbone:
            feat = feat.detach()  # freeze backbone so no gradient flows; below, ResNet freezes via requires_grad=False instead of detach() -- both stop gradients, but the latter is cheaper when the backbone is never going to be unfrozen later
        return self.fc(feat)

def train_source(seed, epochs=300, lr=0.001):
    torch.manual_seed(seed)  # seed before constructing the model (Lesson 34/35)
    backbone = Backbone()
    fc = nn.Linear(32, 2)
    opt = torch.optim.Adam(list(backbone.parameters()) + list(fc.parameters()), lr=lr)
    Xt = torch.tensor(X_src).permute(0, 3, 1, 2); yt = torch.tensor(y_src)
    for _ in range(epochs):
        opt.zero_grad()
        loss = F.cross_entropy(fc(backbone(Xt)), yt)
        loss.backward()
        opt.step()
    return backbone

def train_target(backbone, freeze_backbone, seed, epochs=200, lr=0.01, backbone_lr=None):
    torch.manual_seed(seed)
    model = Classifier(backbone, freeze_backbone)
    if freeze_backbone:
        opt = torch.optim.Adam(model.fc.parameters(), lr=lr)
    elif backbone_lr is not None:
        opt = torch.optim.Adam([
            {'params': model.backbone.parameters(), 'lr': backbone_lr},
            {'params': model.fc.parameters(), 'lr': lr},
        ])
    else:
        opt = torch.optim.Adam(model.parameters(), lr=lr)
    Xtr = torch.tensor(X_tgt_train).permute(0, 3, 1, 2); ytr = torch.tensor(y_tgt_train)
    Xte = torch.tensor(X_tgt_test).permute(0, 3, 1, 2); yte = torch.tensor(y_tgt_test)
    for _ in range(epochs):
        opt.zero_grad()
        loss = F.cross_entropy(model(Xtr), ytr)
        loss.backward()
        opt.step()
    with torch.no_grad():
        return (model(Xte).argmax(1) == yte).float().mean().item()

In [ ]:
pretrained = train_source(seed=0)  # pretrain once; the point of transfer learning is reusing this

scratch_accs, frozen_accs, finetune_accs = [], [], []
for seed in range(8):
    scratch_acc = train_target(Backbone(), freeze_backbone=False, seed=seed)
    frozen_acc = train_target(copy.deepcopy(pretrained), freeze_backbone=True, seed=seed)
    finetune_acc = train_target(copy.deepcopy(pretrained), freeze_backbone=False, seed=seed, backbone_lr=0.0003)
    scratch_accs.append(scratch_acc); frozen_accs.append(frozen_acc); finetune_accs.append(finetune_acc)

print(f'{"strategy":>20} {"mean test acc":>16} {"std":>8}')
print(f'{"from scratch":>20} {np.mean(scratch_accs):>15.1%} {np.std(scratch_accs):>8.1%}')
print(f'{"frozen backbone":>20} {np.mean(frozen_accs):>15.1%} {np.std(frozen_accs):>8.1%}')
print(f'{"fine-tuned":>20} {np.mean(finetune_accs):>15.1%} {np.std(finetune_accs):>8.1%}')
print(f'\n(averaged over {len(scratch_accs)} random seeds, each reusing the same 30 target training images)')

Bảng ở trên chỉ cho thấy độ chính xác cuối cùng của mỗi chiến lược. Ảnh động bên dưới theo dõi cả ba đường cong độ chính xác kiểm tra đích qua quá trình huấn luyện (trung bình &plusmn; 1&sigma; qua cùng 8 seed, hiển thị dưới dạng một dải tô bóng): tinh chỉnh leo nhanh nhất và ổn định ở mức cao nhất; mạng xương sống đóng băng mượt mà, ổn định và tập trung chặt chẽ qua các seed; còn huấn luyện từ đầu vẫn nhiễu, kẹt ở mức thấp hơn, và trải rộng thấy rõ hơn qua các seed suốt quá trình.

<img src="../animations/transfer_race_demo.gif" alt="Ảnh động cuộc đua của ba đường cong độ chính xác kiểm tra nhiệm vụ đích qua 200 epoch, mỗi đường có một dải tô bóng cộng-trừ-một-độ-lệch-chuẩn qua 8 seed: huấn luyện từ đầu kẹt ở mức thấp hơn và nhiễu quanh 64 phần trăm với một dải tô bóng rộng phản ánh phương sai cao giữa các seed, mạng xương sống đóng băng mượt mà và ổn định quanh 70 phần trăm với một dải hẹp chặt, và tinh chỉnh leo nhanh nhất để ổn định quanh 77 phần trăm với một dải hẹp tương tự" style="display: block; margin: 0 auto; max-width: 760px;">

Cả hai chiến lược chuyển giao đều vượt rõ rệt việc huấn luyện từ đầu, và chúng cũng *nhất quán* hơn đáng kể (độ lệch chuẩn thấp hơn). Dù nhiệm vụ nguồn chưa bao giờ thấy một con mèo hay một chiếc xe tải, nó đã học được các đặc trưng cấp thấp chung chung (biên, đốm màu, kết cấu) có thể chuyển giao sang nhiệm vụ mới.

Lưu ý rằng tinh chỉnh dùng tốc độ học cho mạng xương sống nhỏ hơn khoảng 30 lần so với phần đầu phân loại. Với chỉ 30 mẫu đích, một cập nhật mạng xương sống lớn hơn sẽ quá khớp 30 ảnh đó, vứt bỏ mọi thứ hữu ích mà nó đã học từ nhiệm vụ nguồn 600 ảnh. Đây là cùng kiểu thất bại quên thảm khốc (catastrophic forgetting) như đường cong quá khớp ở Bài 35, chỉ là áp dụng cho một mạng vốn đã biết điều gì đó từ đầu.

Thêm một chi tiết đáng chú ý: `pretrained` được huấn luyện một lần (`seed=0`) và dùng lại qua cả tám seed trong vòng lặp ở trên. Chỉ phần huấn luyện phía đích (khởi tạo bộ phân loại, quỹ đạo tinh chỉnh) thay đổi theo `seed`, nên các độ lệch chuẩn báo cáo ở trên chỉ đo phương sai từ huấn luyện phía đích, không phải nhiễu trong việc tiền huấn luyện nguồn.

## Trong thực tế: mạng xương sống đã tiền huấn luyện chỉ trong một dòng (In practice: pretrained backbones in one line)
`Backbone` ở trên được huấn luyện từ đầu trên một nhiệm vụ đồ chơi nhằm mục đích minh họa, nhưng học chuyển giao thực tế tận dụng các mạng có sẵn để dùng lại. `torchvision` cung cấp các kiến trúc đã được tiền huấn luyện trên toàn bộ tập dữ liệu ImageNet (Bài 37) — 1,2 triệu ảnh trên 1.000 lớp. Tải một mạng như vậy, sẵn sàng dùng làm bộ trích xuất đặc trưng đóng băng, chỉ là một dòng code:

In [ ]:
import torchvision

resnet = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
n_total = sum(p.numel() for p in resnet.parameters())

resnet.fc = nn.Identity()  # strip the original 1000-class ImageNet head, keep the 512-d features underneath
resnet.eval()
for p in resnet.parameters():
    p.requires_grad = False  # frozen, feature-extraction style — the same strategy as "frozen backbone" above

print(f'total ResNet-18 parameters: {n_total:,}  (all pretrained on ImageNet, none of them touched here)')

ResNet-18 được huấn luyện trên ảnh 224x224 được chuẩn hóa theo trung bình và độ lệch chuẩn theo từng kênh của ImageNet — không phải phép co giãn điểm ảnh `[0, 1]` đơn thuần được dùng ở mọi chỗ khác trong bài này — nên các ảnh đích cần được đổi kích thước và chuẩn hóa lại trước khi đi qua nó. Trích xuất một vector đặc trưng 512 chiều cho mỗi ảnh đích một lần (mạng xương sống bị đóng băng, nên điều này chỉ cần làm một lần, không phải ở mỗi bước huấn luyện), rồi huấn luyện một phần đầu tuyến tính nhỏ bên trên — chính xác là chiến lược mạng xương sống đóng băng ở trên, chỉ với một mạng xương sống rất khác.

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def extract_features(X):
    x = torch.tensor(X).permute(0, 3, 1, 2)
    x = F.interpolate(x, size=224, mode='bilinear', align_corners=False)
    x = (x - IMAGENET_MEAN) / IMAGENET_STD
    with torch.no_grad():
        return torch.cat([resnet(x[i:i + 32]) for i in range(0, len(x), 32)])

feat_train = extract_features(X_tgt_train)
feat_test = extract_features(X_tgt_test)
ytr_t = torch.tensor(y_tgt_train); yte_t = torch.tensor(y_tgt_test)

resnet_accs = []
for seed in range(8):
    torch.manual_seed(seed)
    head = nn.Linear(feat_train.shape[1], 2)
    opt = torch.optim.Adam(head.parameters(), lr=0.01)
    for _ in range(200):
        opt.zero_grad()
        loss = F.cross_entropy(head(feat_train), ytr_t)
        loss.backward()
        opt.step()
    with torch.no_grad():
        resnet_accs.append((head(feat_test).argmax(1) == yte_t).float().mean().item())

print(f'frozen real ImageNet ResNet-18: mean test acc = {np.mean(resnet_accs):.1%}  (+/- {np.std(resnet_accs):.1%}, 8 seeds)')
print(f'(toy frozen backbone above got {np.mean(frozen_accs):.1%})')

In [ ]:
means = [np.mean(frozen_accs), np.mean(resnet_accs)]
labels = ['toy frozen\nbackbone', 'frozen ImageNet\nResNet-18']

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(labels, means, color=['#1f6fb4', '#2e9e44'])
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, m + 0.02, f'{m:.1%}', ha='center',
            fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.08)
ax.set_ylabel('target test accuracy (mean, 8 seeds)')
ax.set_title('same frozen-backbone strategy, a much richer backbone')
plt.show()

Các đặc trưng ImageNet đóng băng đẩy độ chính xác vượt xa cả hai chiến lược đồ chơi ở trên, chỉ dùng một lớp tuyến tính mới được huấn luyện trên cùng 30 ảnh đích — hoàn toàn không huấn luyện mạng xương sống.

**Một lưu ý về phép so sánh này:** một phần khoảng cách đó là một lợi thế không công bằng, không chỉ do tiền huấn luyện phong phú hơn. 1.000 lớp của ImageNet đã bao gồm ảnh mèo và vài loại phương tiện, khác với nhiệm vụ nguồn đồ chơi, vốn chỉ thấy máy bay và ô tô — nên đây không phải là một phép so sánh được kiểm soát hoàn hảo giữa "tiền huấn luyện tổng quát lớn" và "tiền huấn luyện 2 lớp nhỏ bé" một cách tách biệt; một phần lợi thế của ResNet-18 đến từ việc đã trực tiếp thấy các lớp đích trong khi tiền huấn luyện.

Tuy nhiên, điểm định tính vẫn đúng, và đó là lý do thực sự khiến các mạng xương sống tiền huấn luyện trên ImageNet là điểm xuất phát mặc định trong thực tế: một mạng được huấn luyện trên 1,2 triệu ảnh học được các đặc trưng phong phú hơn, có khả năng chuyển giao rộng hơn rất nhiều so với một mạng được huấn luyện trên vài nghìn ảnh của hai danh mục — đó chính xác là thứ mà các kiến trúc ở Bài 37, cùng các tập dữ liệu lớn hơn rất nhiều mà chúng được huấn luyện, được xây dựng để tạo ra.

### Bài tập (Exercises)
1. Thử `backbone_lr=0.01` (tức là không có khác biệt tốc độ học giữa mạng xương sống và phần đầu) trong lệnh gọi tinh chỉnh. Độ chính xác sau tinh chỉnh tốt hơn hay tệ hơn, và điều đó có khớp với lời giải thích về quên thảm khốc ở trên không?
2. Thử đóng băng *phần lớn* mạng xương sống nhưng chỉ tinh chỉnh lớp tích chập cuối cùng của nó (gợi ý: đặt `requires_grad = False` chỉ trên các tham số của `Conv2d` đầu tiên). Kết quả nằm ở đâu so với đóng băng hoàn toàn và tinh chỉnh hoàn toàn?
3. Đổi các lớp của nhiệm vụ đích từ mèo/xe tải sang chó/ô tô — một cặp đích chồng lấn với *miền (domain)* của nhiệm vụ nguồn (phương tiện) còn sát hơn cả mèo/xe tải. Độ chính xác của mạng xương sống đóng băng có cải thiện thêm không, và điều đó có khớp với trực giác rằng chuyển giao hoạt động tốt nhất khi nguồn và đích chia sẻ nhiều cấu trúc thị giác bên dưới hơn không?
4. `resnet.fc = nn.Identity()` giữ đóng băng mọi lớp tích chập của ResNet-18. Hãy thử mở băng chỉ giai đoạn dư cuối cùng của nó (`resnet.layer4`) và tinh chỉnh nó trên 30 ảnh đích với một tốc độ học nhỏ, giống cách mạng xương sống đồ chơi đã được tinh chỉnh ở trên. Nó có vượt qua kết quả ResNet-18 đóng băng hoàn toàn không, hay 30 ảnh là quá ít để cập nhật an toàn ngay cả một phần nhỏ của 11 triệu tham số?